# Building training datasets for scTRP

This notebook walks through turning a 10x output folder plus a reactivity annotation into the
two inputs the training pipeline needs:

1. a **log1p `.h5ad`** (expression + `obs['cdr3_aa']` + `obs['reactivity']`), and
2. an **ESM2-encoded `.tar`** (`*_esm2encoded.tar`), passed to `--train_data_path_simclr` / `--val_data_path_simclr`.

| Step | What it does | Script | Output |
|---|---|---|---|
| 1 | QC, normalise, log1p, HVG, attach CDR3 + reactivity | `prep_log1p_h5ad.py` | `{SAMPLE}_log1p.h5ad` |
| 2 | Tokenize each cell with the scGPT vocabulary | `tokenize_h5ad.py` | `tokenized_data/*.npz` |
| 3 | Pack the npz files into a webdataset `.tar` | (python cell below) | `{SAMPLE}.tar` |
| 4 | Encode CDR3 sequences with ESM2 | `encode_seq_with_pretrained_ESM2.py` | `{SAMPLE}_esm2encoded.tar` |

Run this notebook from inside `data_prep/` (the cells call the scripts by relative path).
Steps 1–2 and 4 need the scverse / scGPT / ESM2 environment (see the main README, Installation).
Step 4 needs a GPU.

> **Important: provide the `reactivity` annotation in Step 1.**
> `prep_log1p_h5ad.py` requires a CSV whose **index is the cell barcode** (e.g. `AAACCTGAGAAGGCCT-1`)
> and which has a column named **`reactivity`** with values `0` / `1`. Cells without a label are dropped.
> Without this column the script stops before writing anything.

## Configuration

Edit these paths once. Every step below reads from here, so the same notebook works for any sample.
To process several samples, change `SAMPLE_NAME` and the input paths and re-run from Step 1.

In [ ]:
import os
import tarfile

import numpy as np

SAMPLE_NAME    = "TenX_CD8T_Donor1_antigen"      # used for file/folder names and as the npz id suffix
INPUT_10X_DIR  = "/path/to/10x_output_dir/"      # contains *feature_bc_matrix.h5 and *_t_*_contig_annotations.csv
REACTIVITY_CSV = "/path/to/reactivity.csv"       # REQUIRED: index = barcode, column 'reactivity' (0/1)
GENE_LIST_CSV  = None                            # optional CSV of gene symbols; None = 2000 HVGs
WORK_DIR       = "/path/to/work_dir"             # all outputs are written under here
SCGPT_MODEL_DIR = "/path/to/scgpt_human"         # contains vocab.json, args.json, best_model.pt
N_GENES_CUT    = 5000
PCT_MT_CUT     = 20
MAX_LENGTH     = 1200
SCGPT_PATH     = None                       # scGPT source dir (contains scgpt/); None = SCGPT_PATH env var or known paths

SAMPLE_DIR  = os.path.join(WORK_DIR, SAMPLE_NAME)
H5AD_PATH   = os.path.join(SAMPLE_DIR, f"{SAMPLE_NAME}_log1p.h5ad")
TOKEN_DIR   = os.path.join(SAMPLE_DIR, "tokenized_data")
TAR_PATH    = os.path.join(WORK_DIR, f"{SAMPLE_NAME}.tar")
ENC_TAR     = os.path.join(WORK_DIR, f"{SAMPLE_NAME}_esm2encoded.tar")   # written by Step 4

os.makedirs(SAMPLE_DIR, exist_ok=True)
print("log1p h5ad  :", H5AD_PATH)
print("tokenized   :", TOKEN_DIR)
print("tar         :", TAR_PATH)
print("ESM2 tar    :", ENC_TAR)

## Step 1 — Build the log1p `.h5ad`

`prep_log1p_h5ad.py` runs the QC / normalisation / HVG steps from `load_TCR_RNA_prep.py` and writes
an AnnData whose `obs` holds `cdr3_aa` and `reactivity`.

**Bash equivalent:**

```bash
python data_prep/prep_log1p_h5ad.py \
    --input_dir      /path/to/10x_output_dir/ \
    --reactivity_csv /path/to/reactivity.csv \
    --output_h5ad    /path/to/work_dir/SAMPLE/SAMPLE_log1p.h5ad \
    --n_genes_cut 5000 --pct_mt_cut 20 \
    [--gene_list /path/to/genes.csv]
```

In [ ]:
gene_arg = f"--gene_list {GENE_LIST_CSV}" if GENE_LIST_CSV else ""
!python prep_log1p_h5ad.py --input_dir {INPUT_10X_DIR} --reactivity_csv {REACTIVITY_CSV} --output_h5ad {H5AD_PATH} --n_genes_cut {N_GENES_CUT} --pct_mt_cut {PCT_MT_CUT} {gene_arg}

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(H5AD_PATH)
print(adata)
print(adata.obs[["cdr3_aa", "reactivity"]].head())
print(adata.obs["reactivity"].value_counts())

## Step 2 — Tokenize into per-cell `.npz` files

`tokenize_h5ad.py` maps each cell's non-zero genes to scGPT vocabulary ids, applies binning, and
writes one npz per cell. The cell id is `{barcode}-{SAMPLE_NAME}`, which keeps ids unique when
several samples are later combined. Each npz also stores `seq` (the CDR3), which Step 4 reads.

**Bash equivalent:**

```bash
python data_prep/tokenize_h5ad.py \
    --input_h5ad      /path/to/work_dir/SAMPLE/SAMPLE_log1p.h5ad \
    --output_dir      /path/to/work_dir/SAMPLE/tokenized_data/ \
    --scgpt_model_dir /path/to/scgpt_human \
    --prefix          SAMPLE \
    --max_length      1200
```

In [ ]:
!python tokenize_h5ad.py --input_h5ad {H5AD_PATH} --output_dir {TOKEN_DIR} --scgpt_model_dir {SCGPT_MODEL_DIR} --prefix {SAMPLE_NAME} --max_length {MAX_LENGTH}

In [ ]:
npz_files = [f for f in os.listdir(TOKEN_DIR) if f.endswith(".npz")]
print(len(npz_files), "npz files; example:", npz_files[:2])

## Step 3 — Pack the npz files into a `.tar`

Step 4 reads its input as a webdataset tar. The archive contains the `tokenized_data/` folder.

This is the same packing as the code you provided, with one change: the folder path is passed through
`rstrip("/")` first. `os.path.basename("…/tokenized_data/")` returns an empty string, so with a trailing
slash the archive would have no folder name.

**Bash equivalent:**

```bash
tar -cf /path/to/work_dir/SAMPLE.tar -C /path/to/work_dir/SAMPLE tokenized_data
```

In [ ]:
folder = TOKEN_DIR.rstrip("/")
with tarfile.open(TAR_PATH, "w") as tar:
    tar.add(folder, arcname=os.path.basename(folder))
print("wrote", TAR_PATH, f"({os.path.getsize(TAR_PATH) / 1e6:.1f} MB)")

## Step 4 — ESM2-encode the CDR3 sequences

`encode_seq_with_pretrained_ESM2.py` reads the tar from Step 3. For each npz it adds an `esm2_emb`
array (shape `[1, 1280]`). It writes a new folder `{SAMPLE}_esm2encoded/` and packs it into
`{SAMPLE}_esm2encoded.tar`. That archive is the file to pass to the training pipeline.

The script needs a GPU. It finds scGPT from `--scgpt_path` (or the `SCGPT_PATH` environment variable).
The ESM2 encoder is imported from the repo's `utils/simclr_esm2_functions.py`.

**Bash equivalent:**

```bash
python data_prep/encode_seq_with_pretrained_ESM2.py \
    --data_url  /path/to/work_dir/SAMPLE.tar \
    --run_mode  do_seq_only \
    [--scgpt_path /path/to/scGPT]   # optional if SCGPT_PATH env var is set
# -> /path/to/work_dir/SAMPLE_esm2encoded.tar
```

In [ ]:
path_arg = f"--scgpt_path {SCGPT_PATH}" if SCGPT_PATH else ""
!python encode_seq_with_pretrained_ESM2.py --data_url {TAR_PATH} --run_mode do_seq_only {path_arg}


In [ ]:
assert os.path.exists(ENC_TAR), f"missing {ENC_TAR}"
with tarfile.open(ENC_TAR) as tar:
    names = [n for n in tar.getnames() if n.endswith(".npz")]
    print(len(names), "encoded cells")
    with np.load(tar.extractfile(names[0])) as z:
        print("keys:", list(z.keys()))
        print("esm2_emb shape:", z["esm2_emb"].shape)

## Use the output in training

Pass the encoded tar to the training scripts. This example uses Mode A from the main README:

```bash
python scripts/train_simclr.py \
    --train_data_path        /path/to/train.h5ad \
    --val_data_path          /path/to/val.h5ad \
    --test_data_path         /path/to/test.h5ad \
    --train_data_path_simclr /path/to/work_dir/SAMPLE_esm2encoded.tar \
    --val_data_path_simclr   /path/to/val_esm2encoded.tar \
    ...
```

Each cell's id in the tar is `{barcode}-{SAMPLE_NAME}`. Check that the `.h5ad` used for training
has obs names that match these ids.